# Tool使用的概述

## 1、工具调用方式

### 1.1 方式1：直接调用

In [2]:
from langchain_core.tools import tool

@tool
def get_werther(city: str) -> str:
    """
    获取指定的天气信息

    :param
        city:城市名称，如“北京”、“上海”，“南昌”
    :return:
        天气信息字符串
    """
    # 你的实现
    return city + "晴天，温度15℃"

In [3]:
get_werther.invoke({"city":"北京"})

'北京晴天，温度15℃'

### 1.2 方式2：基于模型进行调用

In [4]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 1、提供大模型
load_dotenv(override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    # temperature=1.5,
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
    # max_tokens=10,
)

In [11]:
from langchain_core.tools import tool

# 定义工具
@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气"""
    # 你的实现
    return "晴天，温度15℃"

# 绑定工具
model_with_tools = model.bind_tools([get_werther])

# AI可以决定是否调用工具
response = model_with_tools.invoke("仲恺天气如何？")
# response = model_with_tools.invoke("2 + 3 = ?")

# 检查AI是否要调用工具
if response.tool_calls:
    print("AI想调用工具：",response.tool_calls)
else:
    print("AI直接回答：", response.content)

AI想调用工具： [{'name': 'get_werther', 'args': {'city': '仲恺'}, 'id': 'call_db4e728f98be42ba8f19af11', 'type': 'tool_call'}]


## 2、从Message流转看工具的调用

In [12]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 1、提供大模型
load_dotenv(override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    # temperature=1.5,
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
    # max_tokens=10,
)

In [22]:
from langchain_core.tools import tool
from langchain.messages import HumanMessage, ToolMessage
from rich import print as rprint

@tool
def get_weather(city: str):
    """获取天气的工具"""
    return f"{city}天气晴朗"

# 将模型和工具绑定
model_with_tools = model.bind_tools([get_weather])

# 声明一个消息列表
messages =  [
    HumanMessage("今天仲恺天气如何？")
]

# 模型生成调用工具请求
response = model_with_tools.invoke(messages)

# 添加AIMessage到消息列表中
messages.append(response)

# rprint(response)

tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather":
        # 大模型和Agent的主要区别在于：大模型不会主动调用工具，所有这时候我们需要主动让工具调用
        # 返回的是ToolMessage类消息，添加到消息列表中
        tool_response = get_werther.invoke(tool_call)
        print(type(tool_response))
        messages.append(tool_response)

print("=================================> Messages <==================================")
for message in messages:
    message.pretty_print()
print("=================================> Messages <==================================")
final_response = model_with_tools.invoke(messages)
print(type(final_response))
print(f"final_response:\n{final_response}")

<class 'langchain_core.messages.tool.ToolMessage'>
=================================> Messages <==================================
================================ Human Message =================================

今天仲恺天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_9af7db2a53a34a93a97379fb)
 Call ID: call_9af7db2a53a34a93a97379fb
  Args:
    city: 仲恺
================================= Tool Message =================================
Name: get_werther

仲恺晴天，温度15℃
=================================> Messages <==================================
<class 'langchain_core.messages.ai.AIMessage'>
final_response:
content='今天仲恺的天气是晴天，气温为15℃。天气不错，祝您有美好的一天！' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 140, 'prompt_tokens': 323, 'total_tokens': 463, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 114, 'rejected_prediction_tokens